In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binom
from scipy.stats import poisson

Load the dataset

In [ ]:
#Trips, Nights and Average Length of Stay of Foreign Resident Overnight Visitors
#dataset downloaded from 
# ITM05 https://data.cso.ie/#

tourism=pd.read_csv('Length of Stay of Foreign Resident.csv')

head to see the first 5 raws

In [ ]:
tourism.head()

In [ ]:
#to see how many columns and rows it has
tourism.shape

In [ ]:
#to see if we have nulls and how many columns
tourism.info()

In [ ]:
# stadistic properties mean, std, min
tourism.describe()

In [ ]:
# I can see what types of data I have, mostly str
tourism.dtypes

In [ ]:
# to see if there are columns missing values
tourism.isnull().sum()

**Exploratory Data Analysis (EDA)**

Before applying any statistical models, an Exploratory Data Analysis was conducted to understand the dataset's structure and quality.

**Data Quality Insights:**

1. Structure & Types: The dataset has 1,320 rows and 5 columns. We have 4 string columns (categorical) and 1 numeric column (VALUE), which is already a float.
2. Missing Values: isnull().sum() confirmed there are no missing values, so no imputation is required for the raw dataset.
3. Data Structure (Wide vs Long): The value_counts() method revealed that VALUE mixes 5 different metrics (trips, nights, percentages, etc.). Mixing these would yield invalid statistics, which justified filtering by Statistic Label for all subsequent analyses.
4. Distribution & Outliers: Using boxplots and the IQR method, we identified 14 outliers in the Number of Overnight Trips. The histogram of Average Length of Stay showed a strong right-skew, indicating non-normality.
* Conclusion: The EDA revealed that the raw data requires careful filtering (by metric) before analysis. For Machine Learning, the dataset will need to be pivoted (wide format) and the categorical variables (Accommodation Type, Month) will need encoding.

# Correlation

"For the correlation analysis and subsequent ML phases, the dataset was pivoted to a wide format to separate the metrics".

In [ ]:
# Pivot dataset to wide format
ml_df = tourism.pivot_table(
    index=['Month', 'Main Accommodation Type'],
    columns='Statistic Label',
    values='VALUE'
).reset_index()

# Rename columns using a dictionary
ml_df = ml_df.rename(columns={
    'Average Length of Stay of Foreign Visitors': 'Avg_LOS',
    'Number of Nights by Foreign Visitors': 'Nights',
    'Number of Overnight Trips by Foreign Visitors': 'Trips',
    'Percentage of Nights by Foreign Visitors': 'Pct_Nights',
    'Percentage of Overnight Trips by Foreign Visitors': 'Pct_Trips',
    'Main Accommodation Type': 'Accommodation'
})

# Check the column names and the first row to verify the fix
print("Column names:", ml_df.columns.tolist())
print(ml_df.head(1))

In [ ]:

# 1. Select only the numeric metric columns (exclude text and encoded columns)
corr_metrics = ml_df[['Trips', 'Pct_Trips', 'Nights', 'Pct_Nights', 'Avg_LOS']]

# 2. Calculate Spearman correlation (robust to non-normal data)
corr_matrix = corr_metrics.corr(method='spearman')

# 3. Plot the Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Spearman Correlation Matrix of Tourism Metrics')
plt.tight_layout()
plt.show()

# 4. Display the correlation matrix numerically
print("Spearman Correlation Matrix:")
display(corr_matrix.round(2))

**Correlation Analysis**

To understand the relationships between the five tourism metrics, a Spearman correlation matrix was generated on the pivoted dataset (`ml_df`). Spearman was chosen because the data is highly skewed and not normally distributed.

**Insights from the Heatmap:**
- **Strong Positive Correlation:** The *Number of Trips* and *Number of Nights* show a strong positive correlation (0.93). Months with more visitors logically generate more nights.
- **Weak/Moderate Correlation:** *Average Length of Stay* (Avg_LOS) shows a weak or negative correlation with both Trips (-0.34) and Nights (-0.07). This indicates that high volume (many trips) does not necessarily mean longer stays. Volume and duration are driven by different factors.
- **Data Leakage Prevention:** Although the correlation between Avg_LOS and the others is weak, *Avg_LOS* is mathematically derived from *Nights / Trips*. Therefore, including *Avg_LOS*, *Nights*, and *Trips* in the Machine Learning features would allow the model to reconstruct the answer, causing severe data leakage. These variables are excluded from the models.

**Source:** pandas Documentation. "pandas.DataFrame.corr".
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html
**How I found it:** I searched the official pandas documentation for correlation matrix calculations.
**Quality evaluation:** Official pandas documentation, maintained by the pandas development team. It is the primary source for correlation analysis.
**Verification:** I ran the code and confirmed that the output matrix matches the visual heatmap and correctly highlights the relationships between the metrics.

### Time Series Analysis (Seasonality)

In [ ]:


# Filter the dataset to isolate total overnight trips
total_trips = tourism[
    (tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors") & 
    (tourism["Main Accommodation Type"] == "All types of accommodation")
].copy()

# Convert the 'Month' column to datetime
total_trips['Date'] = pd.to_datetime(total_trips['Month'], format='%Y %B')

# Generate the time series plot
plt.figure(figsize=(14, 6))
sns.lineplot(data=total_trips, x='Date', y='VALUE', marker='o', color='steelblue', linewidth=2)

# Formatting
plt.title('Overnight Trips by Foreign Visitors (2023-2026)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Trips (Thousands)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

**Visual Insight: Seasonality (Time Series Plot)**

The line plot displays the evolution of total overnight trips over the analyzed period. This visualization was created to identify seasonal patterns. 

**Tufts Principles Justification:**
The graph adheres to Tufte Principles by displaying the data accurately without distortion (using raw trip counts on the y-axis) and making the data accessible (clear titles, rotated x-axis for readability).

**Insight Gained:** 
The graph clearly reveals the strong seasonal autocorrelation of Irish tourism. Trips peak dramatically during the summer months (July and August) and drop significantly in winter. This insight is critical, as it justifies why the discrete distributions (Binomial and Poisson) have limitations due to their independence assumptions, and it directly informed the creation of the target variable for the Machine Learning models.

**Source:** Pandas Documentation. "pandas.to_datetime".
https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
**How I found it:** Searched the official pandas reference for converting string dates to datetime objects for plotting.
**Quality evaluation:** Primary source for pandas functions.
**Verification:** The code correctly converted the "Month" column and the resulting plot displayed a chronological timeline.

# Measure of central tendency

In [ ]:
# To show the first 5 raws and all columns
tourism.head()

In [ ]:
#Here I noticed that my column statistic label has 5 different metrics with 264 values each
# so i have percentages included so it will not be accurate if I get statistics mixed
print(tourism["Statistic Label"].value_counts())

In [ ]:
#mean
# In my dataset my column value has five metrics, so if apply all
# mixed Ill be getting a wrong mean, so I decided to apply only to one 
# for this case will be trips = Number of Overnight Trips by Foreign Visitors

# Mean of the value if I apply only to the column value which is wrong
mean=tourism["VALUE"].mean()
print (f'The average Value is: {mean}')

Source: pandas Documentation. "pandas.Series.mean". https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.mean.html

How I found it: I searched the official pandas API reference for the mean() method to confirm its default behaviour (arithmetic mean, skipping NaN values).

Quality evaluation: Official pandas documentation, maintained by the pandas development team. It is the primary and authoritative source for pandas methods.

Verification: I ran the code and confirmed that the output is the arithmetic mean of the filtered VALUE column, matching the documented behaviour.

In [ ]:
# If I apply only to Trips Ill be separating only to that specific metric inside my column
# statistic label
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]
print(f"The average Value of Number of Overnight Trips by Foreign Visitors:   {trips['VALUE'].mean():.2f}")

In [ ]:
#median only for trips
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]
print(f"The median Value of Number of Overnight Trips by Foreign Visitors:   {trips['VALUE'].median():.2f}")

In [ ]:
# mode, same as for the rest
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]
mode = trips["VALUE"].mode()
print(mode)
#The mode of Number of Overnight Trips was not informative: 
#out of 264 observations, 247 were unique values, producing 
#17 tied values as the mode. As this is a continuous variable, 
#the mode is not a representative measure; mean and median are reported instead as the 
#main measures of central tendency

In [ ]:
# I found this code to get statistics information separated by groups**Source:** pandas Documentation. "pandas.core.groupby.DataFrameGroupBy.describe".

tourism.groupby('Statistic Label')['VALUE'].describe()


Source: pandas Documentation. "pandas.core.groupby.DataFrameGroupBy.describe". https://pandas.pydata.org/pandas-docs/version/2.2/reference/api/pandas.core.groupby.DataFrameGroupBy.describe.html

How I found it: I searched the official pandas API reference for the groupby().describe() method to confirm that it computes descriptive statistics per group independently.

Quality evaluation: Official pandas documentation, maintained by the pandas development team. It is the primary and authoritative source for pandas methods.

Verification: I ran the code and confirmed that the output contains one set of descriptive statistics per Statistic Label, matching the documented behaviour of grouping before describing.

In [ ]:
# to check wether the data has outliers or not
#boxplot
#the problem here is that i have many types of statistic labels such as percentaje o lenght of stay
plt.boxplot(tourism["VALUE"])
plt.show()
# this graphic is not accurate because my units are differents

In [ ]:
# I am using seaborn as it separates statistic labels by metrics, which i have 5 diferents
plt.figure(figsize=(12, 6))
sns.boxplot(data=tourism, x='Statistic Label', y='VALUE')
plt.yscale('log') #this is to change the dimension to have similar for every metric
plt.xticks(rotation=30, ha='right')
plt.title('Distribution of VALUE by Statistic Label')
plt.ylabel('VALUE (log scale)')
plt.tight_layout()
plt.show()

In [ ]:
#measure of dispersion
#to do for all of the five metrics at the same time I am using the next code
summary = tourism.groupby("Statistic Label")["VALUE"].agg(
    Mean="mean",
    Median="median",
    Std="std",
    Min="min",
    Max="max",
    Count="count",
    #Range = max-min
)
summary["Range"] = summary["Max"] - summary["Min"]
print(summary)

The table above shows the central tendency and dispersion measures for all five metrics. By grouping by Statistic Label, we avoid mixing incompatible units (thousands, percentages, and nights per trip).

A key insight is the difference between the Mean and Median for Number of Overnight Trips (Mean: 181.5 vs Median: 92.4). This confirms the positive skewness observed in the histogram, where a few high-demand summer months pull the mean upwards. The Range also highlights that Number of Nights has the highest absolute dispersion (range of 7,308.9), which is expected given the scale of the data.

In [ ]:
#min of trips
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]

print(f"The minimum value of Number of Overnight Trips by Foreign Visitors is: {trips['VALUE'].min():.2f}")

In [ ]:
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]

print(f"The maximun value of Number of Overnight Trips by Foreign Visitors is: {trips['VALUE'].max():.2f}")

In [ ]:
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]

max_val = trips["VALUE"].max()
min_val = trips["VALUE"].min()
value_range = max_val - min_val

print(f"Max:   {max_val:.2f}")
print(f"Min:   {min_val:.2f}")
print(f"Range: {value_range:.2f}")

In [ ]:
#variance
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]

variance = trips["VALUE"].var()
print(f"Variance: {variance:.2f}")

In [ ]:
# Standard deviation
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]
std= trips["VALUE"].std()
std

In [ ]:
# percentile
"""
Q1 ---> quartile 1 --> 25th percentile
Q3 --> quartile 2 --> 75th percentile
IQR = Q3 - Q1
"""
# again here I am using only one metric
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]

Q1 = trips["VALUE"].quantile(0.25)
Q2 = trips["VALUE"].quantile(0.50)
Q3 = trips["VALUE"].quantile(0.75)
IQR = Q3 - Q1
IQR, Q1, Q2, Q3


In [ ]:
#outliers using percentiles
trips = tourism[tourism["Statistic Label"] == "Number of Overnight Trips by Foreign Visitors"]

Q1 = trips["VALUE"].quantile(0.25)

Q3 = trips["VALUE"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 -1.5*IQR
upper_bound = Q3+ 1.5*IQR

outliers = trips[(trips["VALUE"] < lower_bound) | (trips["VALUE"] > upper_bound )]
print ('Lower bound: ', lower_bound)
print ('Upper bound: ', upper_bound)
print (' Number of outliers ', len(outliers))
print (' Number of outliers ', outliers)
outliers

## Normal Distribution

In [ ]:
# first i am filtering the metric average long of stay of foreign visitors because it is the most
# most continuous variable in the dataset

In [ ]:
# los = long of stay
los = tourism[tourism["Statistic Label"] == "Average Length of Stay of Foreign Visitors"]["VALUE"]

print(f"Number of null values: {los.isnull().sum()}")

## To show if the column average length of stay is normally distributed I visualize


In [ ]:
sns.histplot(los, kde=True)
plt.title("Distribution of Average Length of Stay")
plt.xlabel("Nights per trip")
plt.ylabel("Frequency")
plt.show()


the data is not normally distributed. In a normal distribution, the curve would be symmetric around the mean, with equal tails on both sides.

Empirical rule for normal distribution

In [ ]:
mean = los.mean()
std = los.std()

within1 = ((los >= mean - std) & (los <= mean + std)).mean() * 100
within2 = ((los >= mean - 2*std) & (los <= mean + 2*std)).mean() * 100
within3 = ((los >= mean - 3*std) & (los <= mean + 3*std)).mean() * 100

print(within1,within2,within3)


check normal distribution using z - score

In [ ]:
from scipy import stats
z_scores=stats.zscore(los)
z_scores.min(),z_scores.max()

In [ ]:
##QQ plot, its another way to know if my column is normally distributed
stats.probplot(los,dist='norm',plot=plt)
plt.show()

In [ ]:
#again this graphic shows that is not normal distribution, because the orange line is not together with the blue one and there
# are some points on the end

In [ ]:
from scipy.stats import skew, kurtosis, shapiro

print(f"Skewness: {skew(los):.2f} | Excess kurtosis: {kurtosis(los):.2f} | Shapiro p: {shapiro(los).pvalue:.2e}")

log_los = np.log(los)
print(f"After log: skewness {skew(log_los):.2f} | Shapiro p: {shapiro(log_los).pvalue:.2e}")

sns.histplot(log_los, kde=True)
plt.title('Log of Average Length of Stay')
plt.xlabel('log(Nights per trip)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Binomial Distribution

# 1. Filter to one row per month (All types of accommodation)
monthly = trips[trips["Main Accommodation Type"] == "All types of accommodation"].copy()
monthly["Year"] = monthly["Month"].str[:4]

# 2. Define success (trips > 619)
threshold = 619
monthly["success"] = (monthly["VALUE"] > threshold).astype(int)

# 3. Count successes per year
yearly_success = monthly.groupby("Year")["success"].sum()
yearly_total = monthly.groupby("Year")["success"].count()

print("Months above threshold per year:")
print(yearly_success)
print("\nTotal months per year:")
print(yearly_total)

# 4. Exclude 2026 (only has 8 months)
monthly_full = monthly[monthly["Year"] != "2026"]

# 5. Calculate p (estimated probability of success)
p_hat = monthly_full["success"].mean()
print(f"\nEstimated probability (p): {p_hat:.4f}")

# 6. Plot Binomial(12, p)
n = 12
k = np.arange(0, n + 1)

plt.figure(figsize=(10, 5))
plt.bar(k, binom.pmf(k, n, p_hat), alpha=0.6, color='skyblue', label=f'Binomial(n={n}, p={p_hat:.2f})')
plt.axvline(n * p_hat, color='red', linestyle='--', label=f'Expected Mean = {n * p_hat:.1f}')

plt.xlabel("Number of months with Trips > 619 in a year")
plt.ylabel("Probability")
plt.title("Binomial Distribution: High-Demand Months")
plt.xticks(k)
plt.legend()
plt.tight_layout()
plt.show()

Binomial Distribution Analysis

The Binomial distribution models the number of successes in a fixed number of independent trials. In this dataset:

Trial: Each month of the year.
Success: A month where the number of overnight trips exceeds the IQR upper bound threshold (619 thousand trips).
n: Number of months per year (12, except 2026 which has only 8 months of data).
p: Probability of success, estimated from the data.
Observed Data: The year 2026 was excluded from the parameter estimation due to incomplete data. The model predicts the most likely number of high-demand months per year, which closely matches the observed historical data.

Critical Evaluation (Limitations): The Binomial model assumes that trials (months) are independent. In reality, this assumption is violated due to strong seasonal autocorrelation: high-demand months are clustered in the summer (July-August). This means the Binomial distribution underestimates the probability of consecutive high-demand months. Despite this limitation, it provides a useful theoretical baseline for tourism planning.

Source: SciPy Documentation. "scipy.stats.binom". https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html How I found it: I searched the official SciPy documentation for the Binomial probability mass function. Quality evaluation: Official SciPy documentation, maintained by the open-source community. Verification: I ran the code and confirmed that the generated probabilities matched the theoretical expected values.

In [ ]:
#Poisson 

# 1. Exclude "All types" (we want individual accommodation types only)
individual = trips[trips["Main Accommodation Type"] != "All types of accommodation"].copy()

# 2. Define threshold (median of individual types)
threshold_poisson = individual["VALUE"].median()
print(f"Threshold (median): {threshold_poisson:.2f}")

# 3. Count events per month (how many accommodation types exceed the median)
individual["above"] = (individual["VALUE"] > threshold_poisson).astype(int)
events_per_month = individual.groupby("Month")["above"].sum()

print("\nFirst 10 months of events:")
print(events_per_month.head(10))

# 4. Calculate Lambda (average events per month)
lambda_hat = events_per_month.mean()
print(f"\nEstimated Lambda (λ): {lambda_hat:.4f}")

# 5. Plot Poisson vs Observed Data
k_pois = np.arange(0, events_per_month.max() + 2)

plt.figure(figsize=(10, 5))

# Theoretical Poisson
plt.bar(k_pois, poisson.pmf(k_pois, lambda_hat), alpha=0.6, color='lightgreen', label=f'Poisson(λ={lambda_hat:.2f})')

# Observed data (normalized to density)
plt.hist(events_per_month, bins=np.arange(-0.5, events_per_month.max() + 1.5), 
         density=True, alpha=0.4, color='blue', label="Observed Data")

plt.xlabel("Number of accommodation types exceeding the median")
plt.ylabel("Probability / Density")
plt.title("Poisson Distribution: Events per Month")
plt.xticks(k_pois)
plt.legend()
plt.tight_layout()
plt.show()